In [9]:
!pip -q install gradio scikit-learn pandas numpy joblib

In [10]:
%%writefile config.py
MODEL_PATH = "spam_model.joblib"
THRESHOLD = 0.30          # rewrite until the spam probability is below this
CLASSIFY_AT = 0.50        # label an email as spam at or above this
MAX_REWRITE_STEPS = 10

# Dataset artifacts from one inbox. Never suggest changing these.
BLOCKLIST = {"george", "hp", "hpl", "lab", "labs", "650", "857", "415", "85", "cs", "1999"}

Overwriting config.py


In [11]:
%%writefile features.py
import re
import numpy as np
import pandas as pd
import joblib
from config import MODEL_PATH

BUNDLE = joblib.load(MODEL_PATH)
FEATURES = BUNDLE['feature_names']

WORDS = [c.replace('word_freq_', '') for c in FEATURES if c.startswith('word_freq_')]
CHARS = {
    'char_semicolon': ';', 'char_paren': '(', 'char_bracket': '[',
    'char_exclaim': '!', 'char_dollar': '$', 'char_hash': '#',
}
CAP_COLS = ['capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total']

def extract_features(text):
    text = text.strip()
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    n_words = len(tokens)
    n_chars = len(text)

    counts = {}
    for t in tokens:
        counts[t] = counts.get(t, 0) + 1

    row = {}
    for w in WORDS:
        row['word_freq_' + w] = 100 * counts.get(w, 0) / n_words if n_words else 0.0
    for col, ch in CHARS.items():
        row[col] = 100 * text.count(ch) / n_chars if n_chars else 0.0

    runs = [len(r) for r in re.findall(r"[A-Z]+", text)]
    row['capital_run_length_average'] = np.mean(runs) if runs else 0.0
    row['capital_run_length_longest'] = max(runs) if runs else 0.0
    row['capital_run_length_total'] = sum(runs) if runs else 0.0
    for c in CAP_COLS:
        row[c] = np.log1p(row[c])

    return pd.DataFrame([row])[FEATURES]

Overwriting features.py


In [12]:
%%writefile scorer.py
from features import BUNDLE, WORDS, extract_features

MODEL = BUNDLE['model']

def score_features(X):
    return float(MODEL.predict_proba(X)[0, 1])

def spam_probability(text):
    if not text or not text.strip():
        raise ValueError("Email text is empty")
    return score_features(extract_features(text))

def word_impacts(text):
    X = extract_features(text)
    base = score_features(X)
    rows = []
    for w in WORDS:
        col = 'word_freq_' + w
        if X.at[0, col] == 0:
            continue
        X2 = X.copy()
        X2[col] = 0.0
        rows.append((w, base - score_features(X2)))
    return sorted(rows, key=lambda r: r[1], reverse=True)

Overwriting scorer.py


In [13]:
%%writefile rewriter.py
import re
from config import THRESHOLD, MAX_REWRITE_STEPS, BLOCKLIST
from scorer import spam_probability, word_impacts

SYNONYMS = {
    'free': ['complimentary'], 'money': ['funds'], 'credit': ['account'],
    'remove': ['delete'], 'order': ['request'], 'receive': ['get'],
    'business': ['company'], 'mail': ['message'], 'email': ['message'],
    'internet': ['web'], 'people': ['individuals'], 'report': ['summary'],
    'font': ['typeface'], 'all': ['every'], 'data': ['information'],
    'technology': ['tech'], 'project': ['initiative'], 'parts': ['components'],
}

def replace_word(text, target, replacement):
    pattern = re.compile(r'\b' + re.escape(target) + r'\b', re.IGNORECASE)
    def repl(m):
        o = m.group(0)
        if o.isupper() and len(o) > 1:
            return replacement.upper()
        if o[0].isupper():
            return replacement.capitalize()
        return replacement
    return pattern.sub(repl, text)

def only_target_changed(original, rewritten, target, replacement):
    o = re.findall(r"[A-Za-z0-9]+", original)
    r = re.findall(r"[A-Za-z0-9]+", rewritten)
    if len(o) != len(r):
        return False
    for a, b in zip(o, r):
        if a.lower() != b.lower():
            if a.lower() != target or b.lower() != replacement:
                return False
    skel_o = re.sub(r"[A-Za-z0-9]+", "#", original)
    skel_r = re.sub(r"[A-Za-z0-9]+", "#", rewritten)
    return skel_o == skel_r

def improve(text, threshold=THRESHOLD, max_steps=MAX_REWRITE_STEPS):
    current = text
    p = spam_probability(current)
    start = p
    log = []
    stop = "max_steps"
    for _ in range(max_steps):
        if p < threshold:
            stop = "reached_threshold"
            break
        chosen = None
        for word, drop in word_impacts(current):
            if drop <= 0:
                break
            if word in BLOCKLIST:
                continue
            best = None
            for syn in SYNONYMS.get(word, []):
                cand = replace_word(current, word, syn)
                if cand == current or not only_target_changed(current, cand, word, syn):
                    continue
                p_new = spam_probability(cand)
                if p_new < p and (best is None or p_new < best[1]):
                    best = (syn, p_new, cand)
            if best:
                chosen = (word, best)
                break
        if chosen is None:
            stop = "no_improvement"
            break
        word, (syn, p_new, cand) = chosen
        log.append({'word': word, 'replaced_with': syn,
                    'before': round(p, 4), 'after': round(p_new, 4)})
        current, p = cand, p_new
    else:
        if p < threshold:
            stop = "reached_threshold"
    return {'original': text, 'rewritten': current,
            'start_prob': round(start, 4), 'final_prob': round(p, 4),
            'steps': log, 'stop_reason': stop}

Overwriting rewriter.py


In [14]:
%%writefile app.py
import gradio as gr
import pandas as pd
from config import THRESHOLD, CLASSIFY_AT
from scorer import spam_probability, word_impacts
from rewriter import improve

# Baseline test-set results (logistic regression, 842 test emails).
# Update these if the model is retrained.
METRICS = [
    ("Model accuracy", "93.82%"),
    ("Precision score", "92.77%"),
    ("Detection recall", "91.67%"),
    ("F1 benchmark", "92.22%"),
]

STYLE_HTML = """
<style>
:root, .dark, body, gradio-app {
    --body-background-fill: #0b0f19 !important;
    --background-fill-primary: #0b0f19 !important;
    --background-fill-secondary: #111827 !important;
    --block-background-fill: #111827 !important;
    --block-border-color: #1f2937 !important;
    --block-shadow: none !important;
    --block-radius: 16px !important;
    --block-label-background-fill: #111827 !important;
    --block-label-text-color: #94a3b8 !important;
    --block-title-text-color: #cbd5e1 !important;
    --body-text-color: #e2e8f0 !important;
    --body-text-color-subdued: #94a3b8 !important;
    --input-background-fill: #0b0f19 !important;
    --input-border-color: #1f2937 !important;
    --input-border-color-focus: #3b82f6 !important;
    --input-placeholder-color: #475569 !important;
    --color-accent: #3b82f6 !important;
    --link-text-color: #3b82f6 !important;
    --text-sm: 16px !important;
    --text-md: 18px !important;
    --text-lg: 20px !important;
    --button-primary-background-fill: #f97316 !important;
    --button-primary-background-fill-hover: #ea580c !important;
    --button-primary-text-color: #10b981 !important;
}
gradio-app, body, .main, .app, .wrap, .contain, #root {
    background: #0b0f19 !important;
    max-width: none !important;
}
gradio-app { display: block !important; width: 100% !important; }

.gradio-container {
    max-width: 1280px !important; width: 96% !important;
    margin: 40px auto !important; padding: 48px 56px !important;
    box-sizing: border-box !important;
    background: #0b0f19 !important; font-size: 18px !important;
    border: 1px solid #1e293b !important; border-radius: 20px !important;
    box-shadow: 0 10px 40px rgba(0,0,0,0.6) !important;
}
.main, .contain {
    margin-left: auto !important; margin-right: auto !important;
    max-width: none !important; width: 100% !important;
}

.hero {
    background: linear-gradient(135deg, #1e293b, #0f172a) !important;
    border: 1px solid #3b82f6 !important; border-radius: 16px !important;
    padding: 40px !important; text-align: center !important; margin-bottom: 24px !important;
    box-shadow: 0 0 20px rgba(59,130,246,0.25) !important;
}
.hero-title {
    color: #3b82f6 !important; font-size: 52px !important; font-weight: 800 !important;
    letter-spacing: 1px !important; margin: 0 !important; line-height: 1.2 !important;
}
.hero-sub { color: #94a3b8 !important; font-size: 22px !important; margin-top: 12px !important; }

.metric-grid {
    display: grid !important; grid-template-columns: 1fr 1fr !important;
    gap: 20px !important; margin-bottom: 28px !important;
}
.metric-card {
    background: #111827 !important; border: 1px solid #1f2937 !important;
    border-radius: 14px !important; padding: 26px !important; text-align: center !important;
}
.metric-label {
    font-size: 16px !important; color: #9ca3af !important; text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}
.metric-value { font-size: 40px !important; font-weight: 700 !important; color: #10b981 !important; margin-top: 6px !important; }

#input-panel, #results-panel, #impact-panel, #paraphrase-panel {
    background: #111827 !important; border: 1px solid #1f2937 !important;
    border-radius: 18px !important; padding: 28px !important;
}
#input-panel .block, #results-panel .block, #impact-panel .block, #paraphrase-panel .block {
    background: transparent !important; border: none !important; box-shadow: none !important;
}
textarea, input[type="text"] {
    background: #0b0f19 !important; color: #e2e8f0 !important;
    border: 1px solid #1f2937 !important; border-radius: 12px !important;
    font-size: 20px !important; line-height: 1.5 !important; padding: 16px !important;
}
textarea:focus, input:focus {
    border-color: #3b82f6 !important; box-shadow: 0 0 0 2px rgba(59,130,246,0.25) !important;
}
[data-testid="block-label"], .block-label, .block-title {
    font-size: 17px !important;
}
.prose, .prose p, .prose li { font-size: 18px !important; }
.prose h3 { font-size: 32px !important; margin: 0 0 8px 0 !important; }

#button-row { gap: 20px !important; margin-top: 20px !important; }
#check-btn, #check-btn button, #paraphrase-btn, #paraphrase-btn button {
    background: #f97316 !important; background-color: #f97316 !important;
    border-color: #f97316 !important; color: #10b981 !important;
    font-size: 20px !important; font-weight: 700 !important;
    min-height: 60px !important; padding: 14px 32px !important; border-radius: 12px !important;
}
#check-btn button:hover, #paraphrase-btn button:hover {
    background: #ea580c !important; background-color: #ea580c !important;
}

table, th, td {
    background: transparent !important; color: #cbd5e1 !important; border-color: #1f2937 !important;
    font-size: 17px !important;
}
th { color: #94a3b8 !important; }
a { color: #3b82f6 !important; }
footer { display: none !important; }
</style>
"""

REASONS = {
    "reached_threshold": f"✅ The score is now below {THRESHOLD*100:.0f}%.",
    "no_improvement": ("⚠️ No remaining word swap lowers the score further. The rest of the score "
                       "comes from punctuation and capital letters, which word swaps don't change."),
    "max_steps": "⚠️ Stopped after the maximum number of rewrite rounds.",
}

def header_html():
    cards = "".join(
        f'<div class="metric-card"><div class="metric-label">{label}</div>'
        f'<div class="metric-value">{value}</div></div>'
        for label, value in METRICS
    )
    return (
        '<div class="hero">'
        '<div class="hero-title">🛡️ SPAM SHIELD AI v2.0</div>'
        '<div class="hero-sub">Neural Network Guided Phishing &amp; Scam Telemetry</div>'
        '</div>'
        f'<div class="metric-grid">{cards}</div>'
    )

def build_suggestions(result):
    if result['start_prob'] < THRESHOLD:
        return f"The score is already below {THRESHOLD*100:.0f}%. No changes needed."
    lines = [f"**Score:** {result['start_prob']*100:.1f}% → {result['final_prob']*100:.1f}%", ""]
    for s in result['steps']:
        lines.append(f"- Replace **{s['word']}** with **{s['replaced_with']}** "
                     f"({s['before']*100:.1f}% → {s['after']*100:.1f}%)")
    lines += ["", REASONS[result['stop_reason']], "",
              "*Review every suggested change yourself. The model checks word statistics, not meaning.*"]
    return "\n".join(lines)

def analyze(message):
    empty_table = pd.DataFrame(columns=["Word", "Impact (points)"])
    if not message or not message.strip():
        return "Please enter a message.", {"Spam": 0.0, "Legitimate": 0.0}, empty_table

    p = spam_probability(message)
    label = "🚨 Likely spam" if p >= CLASSIFY_AT else "✅ Likely legitimate"
    verdict = f"### {label}\n{p*100:.1f}% spam probability"
    probs = {"Spam": p, "Legitimate": 1 - p}

    impacts = [(w, round(d * 100, 2)) for w, d in word_impacts(message) if d * 100 >= 0.1][:10]
    impact_df = pd.DataFrame(impacts, columns=["Word", "Impact (points)"]) if impacts else empty_table
    return verdict, probs, impact_df

def paraphrase(message):
    if not message or not message.strip():
        return "Please enter a message first.", ""
    result = improve(message)
    if not result['steps']:
        return build_suggestions(result), "No word changes needed."
    return build_suggestions(result), result['rewritten']

def check_handler(message):
    verdict, probs, impacts = analyze(message)
    return verdict, probs, impacts, gr.update(visible=True), gr.update(visible=False)

def paraphrase_handler(message):
    notes, rewrite = paraphrase(message)
    return notes, rewrite, gr.update(visible=False), gr.update(visible=True)

def build_demo():
    with gr.Blocks() as demo:
        gr.HTML(STYLE_HTML)
        gr.HTML(header_html())

        with gr.Column(elem_id="input-panel"):
            message = gr.Textbox(show_label=False, lines=8, placeholder="Paste an email here...")
            with gr.Row(elem_id="button-row"):
                check_btn = gr.Button("Check spam", variant="primary", elem_id="check-btn")
                paraphrase_btn = gr.Button("Paraphrase", variant="primary", elem_id="paraphrase-btn")

        with gr.Column(elem_id="checker-panel", visible=False) as checker_panel:
            with gr.Row():
                with gr.Column(elem_id="results-panel", scale=1):
                    verdict = gr.Markdown()
                    probs = gr.Label(label="Probabilities", num_top_classes=2)
                with gr.Column(elem_id="impact-panel", scale=1):
                    impacts = gr.Dataframe(label="Words pushing toward spam", interactive=False)

        with gr.Column(elem_id="paraphrase-panel", visible=False) as paraphrase_panel:
            notes = gr.Markdown()
            rewrite = gr.Textbox(label="Paraphrased email", lines=6, interactive=False)

        check_btn.click(check_handler, inputs=message,
                        outputs=[verdict, probs, impacts, checker_panel, paraphrase_panel])
        paraphrase_btn.click(paraphrase_handler, inputs=message,
                             outputs=[notes, rewrite, checker_panel, paraphrase_panel])
    return demo

Overwriting app.py


In [15]:
from app import build_demo
build_demo().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c80f650a730aa6a4ea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
